# Read me


Following notebook assumes all downloads of literature search are presented at path `../../data/wos` in .xls form with column naming as provided by Web of Science platform

## Output of this notebook:

- `../../data/partitions`: Raw data that is to be screened is provided in a single `.csv` and their splits into 5 partitions in `.xlsx` form. 
This includes serves are the base raw data over which screening and coding will be applied. 

- `../../data/partitions-mock`: This is sample mock data where a sample of 1000 records is drawn from each of the 5 partitions. These records serve for only code testing purpose

## Steps Applied to convert WoS manual downloads to raw dataset
The raw dataset provided at `../../data/partitions` is passed through these step before creation of paritions.

### 1) Standardize columns: 
Only few selected columns are kept from the downloaded data to keep dataset managable

### 2) Deduplication: 
The data from WOS was downloaded manually in batches of different publication year. Naturally, many records may have been downloaded multiple times. Hence, deduplicaiton is performed using `UT (Unique WOS ID)` field as primary key of each record. All duplicate records are dropped

### 3) Drop records with missing abstracts:
Screening and coding decisions in the systematic map relies heavily on abstract text. For any record, where the abstract is missing, it is dropped.


For each of the step throughout this notebook, `*_metadata.json` is present that explains the outcomes of this notebook when ran last.

Lastly, it is not required to rerun this notebook. Both `../../data/partitions` and `../../data/partitions-mock` are provided as additional materials of this sytematic map. One may only downloaded and place these at correct paths. Notebook exists for reproducability should you need audit the steps.


In [1]:
from pathlib import Path
import re
import json
import tiktoken
from collections import Counter

import pandas as pd
import numpy as np

load_dir = Path('../../data/wos')
output_dir = Path('../../data/partitions')
output_dir_mock = Path('../../data/partitions-mock')

partition_size = 10000        
partition_size_mock = 1000      # For testing purposes, we can use smaller partitions
KEEP_COLUMNS = [
    'Publication Type',
    'Authors',
    'Article Title',
    'Language',
    'Document Type',
    'Author Keywords',
    'Keywords Plus',
    'Abstract',
    'Publisher',
    'Journal Abbreviation',
    'Journal ISO Abbreviation',
    'Publication Date',
    'Publication Year',
    'DOI',
    'DOI Link',
    'WoS Categories',
    'Web of Science Index',
    'Research Areas',
    'Pubmed Id',
    'UT (Unique WOS ID)',
]
output_dir.mkdir(parents=True, exist_ok=True)
output_dir_mock.mkdir(parents=True, exist_ok=True)
summary = {}

print(f'Load directory: {load_dir.resolve()}')
print(f'Output directory: {output_dir.resolve()}')
print(f'Mock output directory: {output_dir_mock.resolve()}')
print(f'Partition size: {partition_size:,} rows per file')
print(f'Mock partition size: {partition_size_mock:,} rows per file')
print(f'Keeping {len(KEEP_COLUMNS)} columns and dropping the rest')


Load directory: /Users/hunain/d/coding-projects/biodiversity/data/wos
Output directory: /Users/hunain/d/coding-projects/biodiversity/data/partitions
Mock output directory: /Users/hunain/d/coding-projects/biodiversity/data/partitions-mock
Partition size: 10,000 rows per file
Mock partition size: 1,000 rows per file
Keeping 20 columns and dropping the rest


In [2]:
files = sorted(load_dir.rglob('*.xls'))
summary['total_files'] = len(files)
print(f'Found {len(files)} .xls files under {load_dir}')
if not files:
    raise FileNotFoundError('No .xls files were found in ../../data/wos')

print('Sample files (first 10):')
for path in files[:10]:
    print(f' - {path}')


Found 2876 .xls files under ../../data/wos
Sample files (first 10):
 - ../../data/wos/2000/savedrecs (1).xls
 - ../../data/wos/2000/savedrecs (10).xls
 - ../../data/wos/2000/savedrecs (11).xls
 - ../../data/wos/2000/savedrecs (12).xls
 - ../../data/wos/2000/savedrecs (13).xls
 - ../../data/wos/2000/savedrecs (14).xls
 - ../../data/wos/2000/savedrecs (15).xls
 - ../../data/wos/2000/savedrecs (16).xls
 - ../../data/wos/2000/savedrecs (17).xls
 - ../../data/wos/2000/savedrecs (18).xls


In [3]:
preferred_engine = 'calamine'  # fastest for .xls if installed
engines_to_try = [preferred_engine, None, 'openpyxl', 'xlrd']

def load_with_fallback(path):
    errors = []
    for engine in engines_to_try:
        kwargs = {} if engine is None else {'engine': engine}
        engine_name = engine or 'auto'
        try:
            df = pd.read_excel(path, **kwargs)
            return df, engine_name
        except Exception as exc:
            errors.append(f"{engine_name}: {exc}")
    raise RuntimeError('; '.join(errors))

loaded_frames = []
failed_files = []
column_signatures = Counter()

for path in files:
    try:
        frame, engine_used = load_with_fallback(path)
        column_signatures[tuple(frame.columns)] += 1
        frame = frame.reindex(columns=KEEP_COLUMNS)
        loaded_frames.append(frame)
    except Exception as exc:
        failed_files.append((path, str(exc)))

summary['loaded_files'] = len(loaded_frames)
summary['failed_files'] = len(failed_files)

print(f"Files attempted: {len(files)} | Loaded: {summary['loaded_files']} | Failed: {summary['failed_files']}")
if failed_files:
    print('Failed files (path and last error):')
    for path, err in failed_files:
        print(f' - {path}: {err}')


Files attempted: 2876 | Loaded: 2876 | Failed: 0


In [4]:
if not loaded_frames:
    raise RuntimeError('No data frames were loaded; please resolve the load errors above.')

print(f'Unique column signatures (ordered): {len(column_signatures)}')
for cols, count in column_signatures.most_common():
    print(f' - {count} file(s): {len(cols)} columns')
    print(f"   Columns: {', '.join(cols)}")

most_common_cols = list(column_signatures.most_common(1)[0][0])
if len(column_signatures) > 1:
    print('Column differences relative to the most common signature:')
    base_set = set(most_common_cols)
    for cols, count in column_signatures.items():
        col_set = set(cols)
        missing = sorted(base_set - col_set)
        extra = sorted(col_set - base_set)
        if missing or extra:
            print(f" - {count} file(s): missing={missing or 'None'}, extra={extra or 'None'}")
else:
    print('All files share the same column order.')

aligned_frames = [frame.reindex(columns=KEEP_COLUMNS) for frame in loaded_frames]
print(f'Aligned all frames to {len(KEEP_COLUMNS)} kept columns.')


Unique column signatures (ordered): 1
 - 2876 file(s): 72 columns
   Columns: Publication Type, Authors, Book Authors, Book Editors, Book Group Authors, Author Full Names, Book Author Full Names, Group Authors, Article Title, Source Title, Book Series Title, Book Series Subtitle, Language, Document Type, Conference Title, Conference Date, Conference Location, Conference Sponsor, Conference Host, Author Keywords, Keywords Plus, Abstract, Addresses, Affiliations, Reprint Addresses, Email Addresses, Researcher Ids, ORCIDs, Funding Orgs, Funding Name Preferred, Funding Text, Cited References, Cited Reference Count, Times Cited, WoS Core, Times Cited, All Databases, 180 Day Usage Count, Since 2013 Usage Count, Publisher, Publisher City, Publisher Address, ISSN, eISSN, ISBN, Journal Abbreviation, Journal ISO Abbreviation, Publication Date, Publication Year, Volume, Issue, Part Number, Supplement, Special Issue, Meeting Abstract, Start Page, End Page, Article Number, DOI, DOI Link, Book DOI, 

In [5]:
combined = pd.concat(aligned_frames, ignore_index=True)
required_columns = ['Article Title', 'Abstract', 'Publication Year', 'UT (Unique WOS ID)']
missing_required = [col for col in required_columns if col not in combined.columns]
if missing_required:
    raise KeyError(f'Missing required columns: {missing_required}')

summary['rows_before_dedup'] = len(combined)
duplicate_rows = int(combined.duplicated(subset=['UT (Unique WOS ID)']).sum())
summary['duplicate_rows'] = duplicate_rows
print(f"Combined rows (before dedup): {summary['rows_before_dedup']:,}")
print(f"Duplicate rows detected (by UT): {duplicate_rows:,}")

combined = combined.drop_duplicates(subset=['UT (Unique WOS ID)']).reset_index(drop=True)
summary['rows_after_dedup'] = len(combined)
print(f"Rows after removing UT-duplicates: {summary['rows_after_dedup']:,}")


Combined rows (before dedup): 2,861,929
Duplicate rows detected (by UT): 435,121
Rows after removing UT-duplicates: 2,426,808


In [6]:
abstract_series = combined['Abstract']
abstract_missing = abstract_series.isna() | abstract_series.astype(str).str.strip().eq('')
abstract_missing_count = int(abstract_missing.sum())
summary['abstract_null_or_empty'] = abstract_missing_count
print(f"Rows with null/empty abstracts: {abstract_missing_count:,}")

combined = combined.loc[~abstract_missing].copy()
summary['rows_after_abstract_filter'] = len(combined)
print(f"Rows after dropping null/empty abstracts: {summary['rows_after_abstract_filter']:,}")


Rows with null/empty abstracts: 12,617
Rows after dropping null/empty abstracts: 2,414,191


In [7]:
combined['Publication Year'] = pd.to_numeric(combined['Publication Year'], errors='coerce')
publication_year_missing = int(combined['Publication Year'].isna().sum())
summary['publication_year_missing'] = publication_year_missing
combined['UT (Unique WOS ID)'] = combined['UT (Unique WOS ID)'].fillna('').astype(str)
combined = combined.sort_values(by=['Publication Year', 'UT (Unique WOS ID)'], ascending=[False, True]).reset_index(drop=True)
summary['final_rows'] = len(combined)
print(f"Rows ready for export: {summary['final_rows']:,}")
print(f"Rows with missing publication year (sorted to bottom): {publication_year_missing:,}")


Rows ready for export: 2,414,191
Rows with missing publication year (sorted to bottom): 0


In [8]:
summary_table = pd.DataFrame([
    ('Total Excel files', summary.get('total_files', 0)),
    ('Files loaded', summary.get('loaded_files', 0)),
    ('Files failed', summary.get('failed_files', 0)),
    ('Rows before dedup', summary.get('rows_before_dedup', 0)),
    ('Duplicate rows', summary.get('duplicate_rows', 0)),
    ('Rows after dedup', summary.get('rows_after_dedup', 0)),
    ('Abstract null/empty', summary.get('abstract_null_or_empty', 0)),
    ('Rows after abstract filter', summary.get('rows_after_abstract_filter', 0)),
    ('Publication year missing', summary.get('publication_year_missing', 0)),
    ('Final rows', summary.get('final_rows', 0)),
])
summary_table.columns = ['Metric', 'Count']
summary_table


,Metric,Count
0,Total Excel files,2876
1,Files loaded,2876
2,Files failed,0
3,Rows before dedup,2861929
4,Duplicate rows,435121
5,Rows after dedup,2426808
6,Abstract null/empty,12617
7,Rows after abstract filter,2414191
8,Publication year missing,0
9,Final rows,2414191


In [9]:
encoding = tiktoken.get_encoding('cl100k_base')
encode = encoding.encode_ordinary
text_series = combined[['Article Title', 'Abstract']].fillna('').astype(str).agg(' '.join, axis=1)
token_counts = pd.Series(np.fromiter((len(encode(t)) for t in text_series), dtype=np.int64))
token_stats = {
    'sum_tokens': int(token_counts.sum()),
    'avg_per_row': float(token_counts.mean()),
    'median_per_row': float(token_counts.median()),
    'percentile_25': float(token_counts.quantile(0.25)),
    'percentile_75': float(token_counts.quantile(0.75)),
}
token_stats_df = pd.DataFrame(list(token_stats.items()), columns=['Metric', 'Value'])
token_stats_df


,Metric,Value
0,sum_tokens,8.395938e+08
1,avg_per_row,3.477744e+02
2,median_per_row,3.370000e+02
3,percentile_25,2.760000e+02
4,percentile_75,4.040000e+02


In [10]:
full_export_path = output_dir / 'wos_all.csv'
combined.to_csv(full_export_path, index=False)
print(f"Saved full combined dataset ({len(combined):,} rows) -> {full_export_path}")


Saved full combined dataset (2,414,191 rows) -> ../../data/partitions/wos_all.csv


In [11]:
if combined.empty:
    print('No rows to partition; nothing written.')
else:
    num_partitions = int(np.ceil(len(combined) / partition_size))
    pad_width = max(2, len(str(num_partitions)))
    saved_files = []
    saved_mock_files = []
    partition_meta = []
    partition_meta_mock = []
    for idx in range(num_partitions):
        start = idx * partition_size
        end = start + partition_size
        chunk = combined.iloc[start:end]
        chunk_token_counts = token_counts.iloc[start:end]
        partition_id = str(idx + 1)
        year_series = pd.to_numeric(chunk['Publication Year'], errors='coerce')
        year_valid = year_series.dropna()
        min_year = int(year_valid.min()) if not year_valid.empty else 'na'
        max_year = int(year_valid.max()) if not year_valid.empty else 'na'
        fname = f"wos_{str(idx + 1).zfill(pad_width)}_{max_year}-{min_year}.xlsx"

        subdir = output_dir / partition_id
        subdir.mkdir(parents=True, exist_ok=True)
        filename = subdir / fname
        chunk.to_excel(filename, index=False)
        chunk_token_sum = int(chunk_token_counts.sum())
        partition_meta.append({
            'partition': idx + 1,
            'file': str(filename),
            'rows': int(len(chunk)),
            'token_sum': chunk_token_sum,
            'min_year': min_year,
            'max_year': max_year,
        })
        saved_files.append(filename)
        print(f"Saved partition {idx + 1} with {len(chunk):,} rows -> {filename} | tokens: {chunk_token_sum:,}")

        mock_subdir = output_dir_mock / partition_id
        mock_subdir.mkdir(parents=True, exist_ok=True)
        mock_sample_size = min(partition_size_mock, len(chunk))
        mock_chunk = chunk.sample(n=mock_sample_size, random_state=idx).sort_index()
        mock_token_sum = int(token_counts.loc[mock_chunk.index].sum())
        mock_filename = mock_subdir / fname
        mock_chunk.to_excel(mock_filename, index=False)
        partition_meta_mock.append({
            'partition': idx + 1,
            'source_partition_file': str(filename),
            'file': str(mock_filename),
            'rows': int(len(mock_chunk)),
            'token_sum': mock_token_sum,
            'sample_size_requested': int(partition_size_mock),
            'min_year': min_year,
            'max_year': max_year,
        })
        saved_mock_files.append(mock_filename)
        print(f"Saved mock partition {idx + 1} with {len(mock_chunk):,} rows -> {mock_filename} | tokens: {mock_token_sum:,}")

    summary_table_records = summary_table.to_dict(orient='records')
    metadata_payload = {
        'summary_table': summary_table_records,
        'partitions': partition_meta,
    }
    metadata_path = output_dir / 'partition_token_metadata.json'
    metadata_path.write_text(json.dumps(metadata_payload, indent=2))
    metadata_payload_mock = {
        'summary_table': summary_table_records,
        'partitions': partition_meta_mock,
    }
    metadata_path_mock = output_dir_mock / 'partition_token_metadata.json'
    metadata_path_mock.write_text(json.dumps(metadata_payload_mock, indent=2))
    print(f"Completed writing {len(saved_files)} partition file(s).")
    print(f"Partition token metadata written to {metadata_path}")
    print(f"Completed writing {len(saved_mock_files)} mock partition file(s).")
    print(f"Mock partition token metadata written to {metadata_path_mock}")


Saved partition 1 with 10,000 rows -> ../../data/partitions/1/wos_001_2026-2026.xlsx | tokens: 3,643,678
Saved mock partition 1 with 1,000 rows -> ../../data/partitions-mock/1/wos_001_2026-2026.xlsx | tokens: 363,939
Saved partition 2 with 10,000 rows -> ../../data/partitions/2/wos_002_2026-2026.xlsx | tokens: 3,640,124
Saved mock partition 2 with 1,000 rows -> ../../data/partitions-mock/2/wos_002_2026-2026.xlsx | tokens: 362,473
Saved partition 3 with 10,000 rows -> ../../data/partitions/3/wos_003_2026-2026.xlsx | tokens: 3,652,378
Saved mock partition 3 with 1,000 rows -> ../../data/partitions-mock/3/wos_003_2026-2026.xlsx | tokens: 363,065
Saved partition 4 with 10,000 rows -> ../../data/partitions/4/wos_004_2026-2026.xlsx | tokens: 3,644,995
Saved mock partition 4 with 1,000 rows -> ../../data/partitions-mock/4/wos_004_2026-2026.xlsx | tokens: 362,797
Saved partition 5 with 10,000 rows -> ../../data/partitions/5/wos_005_2026-2026.xlsx | tokens: 3,631,502
Saved mock partition 5 with